# 64x64 Rectangular-Obstacle Research Workflow

This notebook clones the project repository, generates the approved datasets using the repository implementation, and provides the staged Phase B workflow. It requires no attached or uploaded dataset.

The attached research plan supplies the research specification; the current repository implementations supply the generation and methodology. The OOD pilot and final ID-holdout/sealed-OOD evaluation have separate approval gates, both disabled by default.

Run cells strictly from top to bottom. Do not enable either approval flag until that stage has separate approval. Each setup-cell execution allocates a unique run directory.

## Frozen research specification

- Dataset: 64x64 grids, overlapping axis-aligned rectangular obstacles, seed 20261106, channel order obstacle occupancy / start / goal, Chebyshev endpoint separation at least 24, Dijkstra labels, and the repository approved D1-D6 intervals and per-group split allocations.
- Movement: eight-connected, no diagonal corner cutting, unit transition cost.
- Neural-guidance accumulation: g(start)=0 and g_candidate(v)=g(u)+Phi(v), charging the destination cell. Preserve strict-lower-cost predecessor updates and existing CLOSED handling.
- Heuristic: h(n)=d_Chebyshev(n,goal)+0.001*d_Euclidean(n,goal); f(n)=g(n)+h(n). Uniformly sample K eligible OPEN nodes without replacement, where K_effective=min(K, eligible OPEN size), then expand the sampled node with the lowest current f. Fixed K values are 4, 5, 10, and 16.
- OOD: independent encoder, pooled Gaussian Mahalanobis detector, Ledoit-Wolf shrinkage, ridge regularization, D95 calibrated only on D1-D3 validation/calibration data, and score s=1-exp(-D_M/D95). Adaptive K is K_max-floor(s*(K_max-K_min)), with bounds 4 and 16. OOD changes K only, not Phi or search mechanics.
- Primary training: Neural A* at most 10 epochs on D1-D3; independent contrastive OOD encoder at most 40 epochs. H1/H2/H3 are optional secondary baselines.
- Fixed-K and adaptive-K comparison arms are supplied by the existing benchmark helpers in the separately gated pilot and final evaluation. This notebook does not duplicate the search implementation.



In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys
from datetime import datetime, timezone

WORK_ROOT = Path("/kaggle/working")
REPOSITORY_URL = "https://github.com/name/AI-project-3.git"
PROJECT_ROOT = WORK_ROOT / "AI-project-3"
if not WORK_ROOT.is_dir():
    raise RuntimeError("Expected Kaggle working directory /kaggle/working was not found.")

if PROJECT_ROOT.exists():
    if not (PROJECT_ROOT / ".git").exists():
        raise FileExistsError(f"{PROJECT_ROOT} exists but is not a Git checkout.")
    current_remote = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "remote", "get-url", "origin"],
        check=True, capture_output=True, text=True,
    ).stdout.strip().rstrip("/")
    if current_remote.removesuffix(".git") != REPOSITORY_URL.removesuffix(".git"):
        raise RuntimeError(f"Checkout origin {current_remote!r} differs from {REPOSITORY_URL!r}.")
else:
    subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
required_modules = ("numpy", "torch", "scipy", "sklearn", "matplotlib")
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(
        "Missing Kaggle packages: " + ", ".join(missing)
        + ". Install only missing dependencies; do not replace or downgrade PyTorch."
    )
import numpy as np
import torch
from data.rectangular_dataset import RUN_ID
from experiments.run_paths import create_run_directory

RESULTS_ROOT = PROJECT_ROOT / "results" / RUN_ID
RESULT_ROOT = create_run_directory(RESULTS_ROOT)
DATA_ROOT = RESULT_ROOT / "data"
RUN_VARIANT = RESULT_ROOT.name
commit = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
run_provenance = {
    "repository_url": REPOSITORY_URL, "repository_commit": commit,
    "run_id": RUN_ID, "run_variant": RUN_VARIANT,
    "result_root": str(RESULT_ROOT),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
(RESULT_ROOT / "source_provenance.json").write_text(
    json.dumps(run_provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(json.dumps(run_provenance, indent=2))
print(f"Working directory: {Path.cwd()}")

## 1. GPU runtime verification

Confirm Kaggle allocated a CUDA device. This does not train or evaluate a model. Keep the accelerator set to GPU; do not switch to CPU if the check fails.

In [ ]:
from experiments.rectangle_pipeline import resolve_device, save_environment

if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError(
        "CUDA is unavailable. Enable a Kaggle GPU accelerator, restart the session, "
        "then rerun this cell. CPU fallback is disabled."
    )
DEVICE = resolve_device("cuda")
GPU_PROPERTIES = torch.cuda.get_device_properties(0)
ENVIRONMENT = save_environment(RESULT_ROOT)
print(f"Device: {DEVICE}")
print(f"GPU: {GPU_PROPERTIES.name}")
print(f"GPU memory: {GPU_PROPERTIES.total_memory / (1024 ** 3):.2f} GiB")
print(f"CUDA runtime: {torch.version.cuda}")
print(f"Environment record: {RESULT_ROOT / 'environment.json'}")

## 2. CUDA preflight for the mandatory primary models

Only Neural A* and the independent OOD encoder are mandatory CUDA preflight components. This synthetic forward/backward check reads no dataset and writes no checkpoints. H1/H2/H3 are optional secondary baselines and are not preflight requirements. There is no CPU fallback.

In [ ]:
import json
import torch
from neural_astar.model import NeuralAStarUNet
from ood.feature_extractor import OODFeatureExtractor

if DEVICE != "cuda" or not torch.cuda.is_available():
    raise RuntimeError("Primary-model preflight requires an allocated CUDA device.")
preflight_results = {}
try:
    candidates = (
        ("NeuralAStar", NeuralAStarUNet(),
         torch.rand((1, 2, 64, 64), dtype=torch.float32, device="cuda")),
        ("OODFeatureExtractor", OODFeatureExtractor(),
         torch.rand((2, 3, 64, 64), dtype=torch.float32, device="cuda")),
    )
    for name, model, features in candidates:
        model = model.to("cuda").train()
        optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)
        optimizer.zero_grad(set_to_none=True)
        torch.cuda.reset_peak_memory_stats()
        output = model(features)
        if name == "OODFeatureExtractor":
            output = model.project(output)
        loss = output.float().square().mean()
        loss.backward()
        torch.cuda.synchronize()
        if not any(parameter.grad is not None for parameter in model.parameters()):
            raise RuntimeError(f"{name} produced no CUDA gradients.")
        preflight_results[name] = {
            "forward_backward_passed": True,
            "output_shape": list(output.shape),
            "peak_allocated_bytes": int(torch.cuda.max_memory_allocated()),
            "peak_reserved_bytes": int(torch.cuda.max_memory_reserved()),
        }
        del optimizer, model, features, output, loss
        torch.cuda.empty_cache()
except torch.cuda.OutOfMemoryError as error:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "CUDA preflight ran out of memory for a mandatory primary model. "
        "Stop and inspect the GPU/session allocation; no CPU fallback is used."
    ) from error

preflight_record = {
    "device": DEVICE, "gpu_name": GPU_PROPERTIES.name,
    "mandatory_models": list(preflight_results), "results": preflight_results,
}
(RESULT_ROOT / "cuda_preflight.json").write_text(
    json.dumps(preflight_record, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(json.dumps(preflight_record, indent=2))

## 3. Generate and validate every approved dataset split

The existing generator creates training, validation/calibration, ID pilot, ID holdout, OOD pilot, and sealed OOD-test splits. Its in-generation validation checks maps, endpoint separation, density intervals, Dijkstra labels, and cross-split disjointness. Outputs are kept under this run's data directory. No pre-generated dataset is attached.

In [ ]:
import hashlib
import json
from data.rectangular_dataset import (
    BASE_SEED, CHANNEL_ORDER, DENSITY_INTERVALS_PERCENT, MAP_SHAPE,
    MIN_CHEBYSHEV_SEPARATION, RectangleGenerationConfig, SPLIT_COUNTS,
    generate_dataset,
)
from algorithms.movement import DEFAULT_COST_CONFIG

expected_splits = {
    "training": {"D1": 300, "D2": 300, "D3": 300},
    "validation_calibration": {"D1": 60, "D2": 60, "D3": 60},
    "id_pilot": {"D1": 20, "D2": 20, "D3": 20},
    "id_holdout": {"D1": 60, "D2": 60, "D3": 60},
    "ood_pilot": {"D4": 20, "D5": 20, "D6": 20},
    "ood_test_sealed": {"D4": 60, "D5": 60, "D6": 60},
}
expected_intervals = {
    "D1": (10, 16), "D2": (17, 23), "D3": (24, 30),
    "D4": (3, 8), "D5": (33, 37), "D6": (38, 42),
}
assert RUN_ID == "64x64_rectangles_v1"
assert BASE_SEED == 20261106 and MAP_SHAPE == (64, 64)
assert MIN_CHEBYSHEV_SEPARATION == 24
assert tuple(CHANNEL_ORDER) == ("obstacle_occupancy", "start_one_hot", "goal_one_hot")
assert SPLIT_COUNTS == expected_splits
assert DENSITY_INTERVALS_PERCENT == expected_intervals
assert DEFAULT_COST_CONFIG.model == "unit_move_count"
assert DEFAULT_COST_CONFIG.prevent_diagonal_corner_cutting is True
if DATA_ROOT.exists() and any(DATA_ROOT.iterdir()):
    raise FileExistsError(f"Refusing to overwrite non-empty run data: {DATA_ROOT}")

generation_report = generate_dataset(DATA_ROOT, config=RectangleGenerationConfig())
assert generation_report["map_count"] == 1560
assert all(generation_report["validation"].values()), generation_report["validation"]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

dataset_hashes = {
    str(path.relative_to(DATA_ROOT)): sha256_file(path)
    for path in sorted(DATA_ROOT.rglob("*")) if path.is_file()
}
dataset_provenance = {
    "run_id": RUN_ID, "run_variant": RUN_VARIANT, "base_seed": BASE_SEED,
    "map_shape": list(MAP_SHAPE), "channel_order": list(CHANNEL_ORDER),
    "split_counts": expected_splits, "density_intervals_percent": expected_intervals,
    "generation_validation": generation_report["validation"],
    "opaque_file_sha256": dataset_hashes,
    "note": "NPZ hashes are byte-only; this cell does not reopen archives.",
}
(RESULT_ROOT / "dataset_provenance.json").write_text(
    json.dumps(dataset_provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(f"Generated maps: {generation_report['map_count']}")
print(f"Dataset directory: {DATA_ROOT}")
print(f"Files hashed: {len(dataset_hashes)}")
print(json.dumps(generation_report["validation"], indent=2, sort_keys=True))

## 4. Train the mandatory primary models

Reuse the project pipeline: Neural A* trains for at most 10 epochs on D1–D3, and the independent contrastive OOD encoder trains for at most 40 epochs. H1/H2/H3 remain optional secondary baselines; the pipeline does not require or train them. Checkpoints, histories, configurations, and summaries go under this run directory.

In [ ]:
from experiments.rectangle_pipeline import train_models

training_summary = train_models(
    DATA_ROOT, RESULT_ROOT, device="cuda", resume=True
)
required_models = {"NeuralAStar", "OODFeatureExtractor"}
assert set(training_summary["models"]) == required_models
assert not {"H1", "H2", "H3"}.intersection(training_summary["models"])
assert training_summary["training_distributions"] == ["D1", "D2", "D3"]
assert training_summary["models"]["NeuralAStar"]["configuration"]["epochs"] == 10
assert training_summary["models"]["OODFeatureExtractor"]["configuration"]["epochs"] == 40
print(json.dumps({
    "training_complete": training_summary["training_complete"],
    "trained_models": sorted(training_summary["models"]),
    "optional_baselines": training_summary["optional_baselines"],
    "summary": str(RESULT_ROOT / "models" / "training_summary.json"),
}, indent=2))

## 5. Fit the OOD detector and calibrate D95

The existing detector uses the independent encoder, pooled Gaussian Mahalanobis distance, Ledoit–Wolf shrinkage, ridge regularization, and D95 calibration from D1–D3 validation/calibration data. The continuous score remains s = 1 − exp(−D_M/D95). This stage does not load OOD pilot or holdout maps.

In [ ]:
from experiments.rectangle_pipeline import calibrate_detector

calibration_summary = calibrate_detector(DATA_ROOT, RESULT_ROOT, device="cuda")
assert calibration_summary["calibration_distributions"] == ["D1", "D2", "D3"]
assert calibration_summary["ood_pilot_used"] is False
assert calibration_summary["sealed_ood_test_used"] is False
assert calibration_summary["d95"] > 0
print(json.dumps({
    "d95": calibration_summary["d95"],
    "calibration_maps": calibration_summary["calibration_sample_count"],
    "calibration_artifact": str(RESULT_ROOT / "calibration" / "detector_calibration.json"),
}, indent=2))

## 6. ID-only integration testing

This validates inference-to-search integration using validation/calibration and ID pilot maps only. It must pass before the OOD pilot gate can proceed. It does not load OOD pilot, ID holdout, or sealed OOD-test maps.

In [ ]:
from experiments.rectangle_pipeline import run_integration_tests

integration_summary = run_integration_tests(DATA_ROOT, RESULT_ROOT, device="cuda")
assert integration_summary["status"] == "passed"
assert integration_summary["maps_used"] == ["validation_calibration", "id_pilot"]
assert integration_summary["ood_pilot_loaded"] is False
assert integration_summary["sealed_ood_test_loaded"] is False
print(json.dumps({
    "status": integration_summary["status"],
    "integration_case_count": integration_summary["integration_case_count"],
    "report": str(RESULT_ROOT / "integration_tests" / "inference_to_search.json"),
}, indent=2))

## 7. OOD pilot — separate approval gate

The pilot uses the 60-map OOD pilot split only after primary training, D95 calibration, and ID-only integration pass. This gate defaults to false. Set OOD_PILOT_APPROVED to true only after separate approval. The existing pilot packager recursively archives the run directory, so this cell temporarily moves the generated data directory outside it while building the pilot ZIP; it restores the data directory in a finally block. This prevents the pilot ZIP from containing holdout or sealed-test archives.

In [ ]:
OOD_PILOT_APPROVED = False

if not OOD_PILOT_APPROVED:
    print("OOD pilot skipped. Keep the gate false until separately approved.")
else:
    import tempfile
    from benchmarks.rectangle_pilot import run_pilot
    from experiments.rectangle_pipeline import cache_pilot_inference

    if integration_summary.get("status") != "passed":
        raise PermissionError("The OOD pilot requires passed ID-only integration.")
    if not (RESULT_ROOT / "models" / "model_ready.json").is_file():
        raise PermissionError("The OOD pilot requires finalized primary models and D95.")
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(DATA_ROOT)

    temp_data_parent = Path(tempfile.mkdtemp(prefix="phase_b_data_", dir=str(WORK_ROOT)))
    relocated_data = temp_data_parent / "generated_data"
    DATA_ROOT.rename(relocated_data)
    try:
        cache_pilot_inference(
            relocated_data, RESULT_ROOT, device="cuda",
            ood_pilot_release_approved=True,
        )
        pilot_summary = run_pilot(relocated_data, RESULT_ROOT, device="cuda")
    finally:
        if relocated_data.exists():
            relocated_data.rename(DATA_ROOT)
        temp_data_parent.rmdir()
    print(json.dumps(pilot_summary, indent=2))
    print(f"Pilot result ZIP: {pilot_summary['results_zip']}")

## 8. Final ID-holdout and sealed-OOD evaluation — separate approval gate

This distinct stage is the only cell that loads ID holdout and sealed OOD-test maps. It runs only after separate approval and completion of the pilot. It reuses the project search arms, worker, path validation, aggregation, and paired-statistics helpers; it does not redefine the search algorithm or OOD mathematics. The complete-run ZIP is generated only after this gate is approved and completed.

In [ ]:
FINAL_EVALUATION_APPROVED = False

if not FINAL_EVALUATION_APPROVED:
    print("Final evaluation skipped; wait for separate holdout/sealed-test approval.")
else:
    import csv
    import hashlib
    import json
    import math
    import zipfile
    from time import perf_counter
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    import torch
    import experiments.rectangle_pipeline as pipeline
    import benchmarks.rectangle_pilot as pilot
    from benchmarks.execution import SearchWorker

    if not (RESULT_ROOT / "pilot" / "pilot_complete.json").is_file():
        raise PermissionError("Complete the separately approved OOD pilot before final evaluation.")
    FINAL_ROOT = RESULT_ROOT / "final_evaluation"
    if FINAL_ROOT.exists() and any(FINAL_ROOT.iterdir()):
        raise FileExistsError(f"Final outputs already exist: {FINAL_ROOT}; use a new run.")
    FINAL_ROOT.mkdir(parents=True, exist_ok=True)

    prior_splits = set(pipeline.ALLOWED_SPLITS)
    try:
        pipeline.ALLOWED_SPLITS = prior_splits | {"id_holdout", "ood_test_sealed"}
        final_samples = {
            "id_holdout": pipeline.load_samples(DATA_ROOT, "id_holdout"),
            "ood_test_sealed": pipeline.load_samples(DATA_ROOT, "ood_test_sealed"),
        }
    finally:
        pipeline.ALLOWED_SPLITS = prior_splits
    assert len(final_samples["id_holdout"]) == 180
    assert len(final_samples["ood_test_sealed"]) == 180
    assert {s.grid_map.distribution for s in final_samples["id_holdout"]} == {"D1", "D2", "D3"}
    assert {s.grid_map.distribution for s in final_samples["ood_test_sealed"]} == {"D4", "D5", "D6"}

    neural, extractor, detector = pipeline._load_finalized_models(RESULT_ROOT, "cuda")
    d95 = float(detector.d95)
    if not math.isfinite(d95) or d95 <= 0:
        raise RuntimeError("Final evaluation requires a valid calibrated D95.")

    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    checkpoint_paths = {
        "neural_astar": RESULT_ROOT / "models" / "neural_astar" / "best_checkpoint.pt",
        "ood_encoder": RESULT_ROOT / "models" / "ood_feature_extractor" / "best_checkpoint.pt",
        "detector": RESULT_ROOT / "detectors" / "mahalanobis_detector.npz",
    }
    caches = {}
    for split, samples in final_samples.items():
        values = {key: [] for key in (
            "phi", "distance", "score", "neural_time", "feature_time",
            "distance_time", "baseline_allocated", "peak_allocated", "peak_reserved",
            "map_ids",
        )}
        for sample in samples:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            values["baseline_allocated"].append(int(torch.cuda.memory_allocated()))
            phi, neural_t = pipeline.predict_guidance(
                neural, sample.grid_map.grid, sample.grid_map.start,
                sample.grid_map.goal, device="cuda",
            )
            if phi.shape != (64, 64) or not np.isfinite(phi).all() or np.any(phi < 0):
                raise ValueError("Invalid Neural A* guidance map during final inference.")
            t0 = perf_counter()
            embedding = pipeline._extract_embedding(extractor, sample, "cuda")
            feature_t = perf_counter() - t0
            t1 = perf_counter()
            distance = float(detector.distance(embedding)[0])
            score = float(pipeline.normalized_ood_score(distance, d95)[0])
            distance_t = perf_counter() - t1
            torch.cuda.synchronize()
            values["phi"].append(phi.astype(np.float32))
            values["distance"].append(distance)
            values["score"].append(score)
            values["neural_time"].append(float(neural_t))
            values["feature_time"].append(float(feature_t))
            values["distance_time"].append(float(distance_t))
            values["peak_allocated"].append(int(torch.cuda.max_memory_allocated()))
            values["peak_reserved"].append(int(torch.cuda.max_memory_reserved()))
            values["map_ids"].append(f"{split}:{sample.grid_map.distribution}:{sample.grid_map.seed}")

        cache_path = FINAL_ROOT / "inference_caches" / f"{split}_inference.npz"
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with cache_path.open("wb") as stream:
            np.savez_compressed(
                stream,
                map_ids=np.asarray(values["map_ids"], dtype=np.str_),
                phi_maps=np.stack(values["phi"]).astype(np.float32),
                mahalanobis_distances=np.asarray(values["distance"], dtype=np.float64),
                continuous_ood_scores=np.asarray(values["score"], dtype=np.float64),
                neural_inference_seconds=np.asarray(values["neural_time"], dtype=np.float64),
                ood_feature_extraction_seconds=np.asarray(values["feature_time"], dtype=np.float64),
                mahalanobis_calculation_seconds=np.asarray(values["distance_time"], dtype=np.float64),
                cuda_baseline_allocated_bytes=np.asarray(values["baseline_allocated"], dtype=np.int64),
                cuda_peak_allocated_bytes=np.asarray(values["peak_allocated"], dtype=np.int64),
                cuda_peak_reserved_bytes=np.asarray(values["peak_reserved"], dtype=np.int64),
            )
        caches[split] = {
            "path": str(cache_path), "sha256": sha256_file(cache_path),
            "maps": len(samples),
            "max_cuda_peak_allocated_bytes": max(values["peak_allocated"]),
            "max_cuda_peak_reserved_bytes": max(values["peak_reserved"]),
        }

    inference_manifest = {
        "run_id": RUN_ID, "run_variant": RUN_VARIANT,
        "loaded_splits": ["id_holdout", "ood_test_sealed"],
        "ood_pilot_completed_first": True, "d95": d95, "caches": caches,
        "checkpoint_sha256": {key: sha256_file(path) for key, path in checkpoint_paths.items()},
    }
    (FINAL_ROOT / "inference_manifest.json").write_text(
        json.dumps(inference_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

    arms = pilot._task_methods()
    expected_rows = sum(len(items) for items in final_samples.values()) * len(arms)
    timeout_seconds = float(json.loads(
        (RESULT_ROOT / "pilot" / "timeout_policy.json").read_text(encoding="utf-8")
    )["per_search_timeout_seconds"])
    raw_path = FINAL_ROOT / "raw" / "search_results.csv"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_fields = list(pilot.RAW_FIELDS) + [
        "cuda_baseline_allocated_bytes", "cuda_peak_allocated_bytes",
        "cuda_peak_reserved_bytes",
    ]
    started = perf_counter()
    rows_written = 0
    with raw_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=raw_fields, extrasaction="ignore")
        writer.writeheader()
        with SearchWorker() as worker:
            for task in pilot._warmup_tasks():
                warmup = worker.run(task, timeout_seconds=timeout_seconds)
                if warmup.response.get("status") != "completed":
                    raise RuntimeError(f"Search warm-up failed: {warmup.response}")
            for split, samples in final_samples.items():
                with np.load(caches[split]["path"], allow_pickle=False) as archive:
                    cache = {key: archive[key].copy() for key in archive.files}
                for index, sample in enumerate(samples):
                    group = str(sample.grid_map.distribution)
                    map_id = f"{split}:{group}:{sample.grid_map.seed}"
                    phi = cache["phi_maps"][index].astype(np.float64, copy=False)
                    distance = float(cache["mahalanobis_distances"][index])
                    score = float(cache["continuous_ood_scores"][index])
                    neural_t = float(cache["neural_inference_seconds"][index])
                    feature_t = float(cache["ood_feature_extraction_seconds"][index])
                    distance_t = float(cache["mahalanobis_calculation_seconds"][index])
                    optimal = float(sample.ground_truth.optimal_path_cost)
                    for arm in arms:
                        task = pilot._search_task_payload(
                            arm, sample=sample, phi=phi, distance=distance,
                            d95=d95, capture_trace=False,
                        )
                        try:
                            execution = worker.run(task, timeout_seconds=timeout_seconds)
                            response = dict(execution.response)
                            roundtrip = float(execution.worker_roundtrip_seconds)
                            restarted = bool(execution.worker_restarted)
                        except BaseException as error:
                            response = {
                                "status": "error", "error_type": type(error).__name__,
                                "error_message": str(error),
                            }
                            roundtrip, restarted = None, False
                        fields = pilot._response_fields(response, arm)
                        fields["worker_restarted"] = restarted
                        metrics = pilot._path_metrics(
                            sample, fields.pop("path"), phi, optimal,
                            use_guidance=arm["method"] in {
                                "neural_astar", "neural_guidance_fixed", "neural_guidance_adaptive"
                            },
                        )
                        fields.update(metrics)
                        if fields["success"] and not fields["valid_path"]:
                            fields.update({
                                "success": False, "invalid_path": True,
                                "failure_reason": "invalid_path", "status": "invalid_path",
                            })
                        path_cost = fields.get("ground_truth_path_cost")
                        if arm["method"] == "dijkstra" and fields["success"]:
                            if path_cost is None or not math.isclose(
                                path_cost, optimal, rel_tol=0.0, abs_tol=1e-12
                            ):
                                raise RuntimeError(f"Dijkstra disagrees with labels for {map_id}.")
                        search_t = fields.get("search_time_seconds")
                        end_to_end = search_t or 0.0
                        if arm["method"] in {"neural_astar", "neural_guidance_fixed"}:
                            end_to_end += neural_t
                        elif arm["method"] == "neural_guidance_adaptive":
                            end_to_end += neural_t + feature_t + distance_t
                        row = {
                            "run_id": RUN_ID, "run_variant": RUN_VARIANT,
                            "split": split, "density_group": group,
                            "actual_density_percent": float(np.count_nonzero(sample.grid_map.grid) * 100 / sample.grid_map.grid.size),
                            "map_id": map_id,
                            "layout_hash": pilot._layout_hash(sample.grid_map.grid),
                            "phi_sha256": pilot._phi_hash(phi),
                            "start": json.dumps(sample.grid_map.start),
                            "goal": json.dumps(sample.grid_map.goal),
                            "sampling_seed": "" if arm.get("seed") is None else str(arm["seed"]),
                            "algorithm": arm["algorithm"], "method": arm["method_name"],
                            "method_family": arm["method_family"], "heuristic": arm["heuristic"],
                            **fields,
                            "goal_counted_as_expanded": True,
                            "search_time_ms": None if search_t is None else 1000.0 * search_t,
                            "worker_roundtrip_seconds": roundtrip,
                            "end_to_end_runtime_seconds": None if search_t is None else float(end_to_end),
                            "dijkstra_optimal_cost": optimal,
                            "raw_mahalanobis_distance": distance,
                            "continuous_ood_score": score, "d95": d95,
                            "model_loading_seconds": None,
                            "neural_inference_seconds": neural_t,
                            "ood_feature_extraction_seconds": feature_t,
                            "mahalanobis_calculation_seconds": distance_t,
                            "cuda_baseline_allocated_bytes": int(cache["cuda_baseline_allocated_bytes"][index]),
                            "cuda_peak_allocated_bytes": int(cache["cuda_peak_allocated_bytes"][index]),
                            "cuda_peak_reserved_bytes": int(cache["cuda_peak_reserved_bytes"][index]),
                        }
                        pilot._write_raw_row(writer, row)
                        rows_written += 1
                    stream.flush()
                    (FINAL_ROOT / "progress.json").write_text(
                        json.dumps({
                            "completed_rows": rows_written, "expected_rows": expected_rows,
                            "last_completed_map": map_id, "status": "running",
                        }, indent=2) + "\n", encoding="utf-8"
                    )

    if rows_written != expected_rows:
        raise RuntimeError(f"Final results incomplete: {rows_written}/{expected_rows}.")
    with raw_path.open("r", newline="", encoding="utf-8") as stream:
        parsed = [pilot._parse_raw_row(row) for row in csv.DictReader(stream)]
    per_map, aggregates = pilot._per_map_and_aggregate(parsed)
    per_map_path = FINAL_ROOT / "per_map" / "per_map_comparison.csv"
    aggregate_path = FINAL_ROOT / "aggregate" / "algorithm_comparison.csv"
    pilot._write_csv(per_map_path, per_map)
    pilot._write_csv(aggregate_path, aggregates)
    statistics = pilot._pilot_statistics(per_map, seed=20261106)
    stats_path = FINAL_ROOT / "aggregate" / "paired_statistics.json"
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    stats_path.write_text(json.dumps(statistics, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    plots_root = FINAL_ROOT / "plots"
    plots_root.mkdir(parents=True, exist_ok=True)
    for split in ("id_holdout", "ood_test_sealed"):
        selected_rows = [row for row in parsed if row["split"] == split]
        for metric, filename, title in (
            ("expanded_nodes", "expanded_nodes.png", "Expanded nodes"),
            ("generated_nodes", "generated_nodes.png", "Generated nodes"),
            ("search_time_ms", "search_runtime_ms.png", "Search-only runtime (ms)"),
            ("ground_truth_path_cost", "path_cost.png", "Path cost"),
        ):
            labels, groups_of_values = [], []
            for method in sorted({row["algorithm"] for row in selected_rows}):
                vals = [
                    float(row[metric]) for row in selected_rows
                    if row["algorithm"] == method and row.get(metric) not in (None, "")
                    and (metric != "ground_truth_path_cost" or str(row["success"]).lower() == "true")
                ]
                if vals:
                    labels.append(method)
                    groups_of_values.append(vals)
            fig, ax = plt.subplots(figsize=(max(11, 0.55 * len(labels)), 6))
            if groups_of_values:
                ax.boxplot(groups_of_values, labels=labels, showfliers=False)
                ax.tick_params(axis="x", rotation=55, labelsize=7)
            ax.set_title(f"{split}: {title}")
            ax.grid(axis="y", alpha=0.25)
            fig.tight_layout()
            fig.savefig(plots_root / f"{split}_{filename}", dpi=160)
            plt.close(fig)

        selected_aggregates = [row for row in aggregates if row["split"] == split]
        core = [
            "neural_guidance_fixed_k4", "neural_guidance_fixed_k5",
            "neural_guidance_fixed_k10", "neural_guidance_fixed_k16",
            "neural_guidance_adaptive_k4_16",
        ]
        densities = sorted(
            {row["density_group"] for row in selected_aggregates},
            key=lambda item: int(item[1:]),
        )
        fig, ax = plt.subplots(figsize=(10, 6))
        for method in core:
            by_density = {row["density_group"]: row for row in selected_aggregates if row["method"] == method}
            ax.plot(
                densities,
                [float(by_density[d]["mean_expanded_nodes"])
                 if d in by_density and by_density[d]["mean_expanded_nodes"] not in (None, "")
                 else np.nan for d in densities],
                marker="o", label=method,
            )
        ax.set_title(f"{split}: fixed-K and adaptive-K expansions by density")
        ax.set_xlabel("Density group")
        ax.set_ylabel("Mean expanded nodes")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(plots_root / f"{split}_fixed_vs_adaptive_k.png", dpi=160)
        plt.close(fig)

    final_summary = {
        "run_id": RUN_ID, "run_variant": RUN_VARIANT,
        "final_evaluation_complete": True,
        "split_map_counts": {name: len(items) for name, items in final_samples.items()},
        "search_arm_count": len(arms), "raw_result_count": rows_written,
        "expected_result_count": expected_rows,
        "search_wall_time_seconds": float(perf_counter() - started),
        "timeout_seconds": timeout_seconds, "d95": d95,
        "paired_comparison_count": len(statistics.get("comparisons", [])),
        "timings_separated": True, "cuda_memory_peaks_recorded": True,
        "id_holdout_loaded": True, "sealed_ood_test_loaded": True,
    }
    (FINAL_ROOT / "final_evaluation_summary.json").write_text(
        json.dumps(final_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    report = [
        "# Final Phase B evaluation report", "",
        f"Run: {RUN_ID}/{RUN_VARIANT}", "",
        "This report covers the separately approved ID-holdout and sealed-OOD final evaluation. The OOD pilot remains in its own stage directory.",
        "",
        f"- D95: {d95}", f"- Search arms: {len(arms)}",
        f"- Raw search rows: {rows_written}",
        f"- ID holdout maps: {len(final_samples['id_holdout'])}",
        f"- Sealed OOD test maps: {len(final_samples['ood_test_sealed'])}",
        f"- Paired comparisons: {len(statistics.get('comparisons', []))}",
        "",
        "## Aggregate results", "",
        "| Split | Group | Method | Maps | Success rate | Mean expanded | Mean generated | Mean search seconds | Mean path cost | Mean path-cost ratio |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for item in aggregates:
        report.append("| " + " | ".join(
            str(item.get(key, "")) for key in (
                "split", "density_group", "method", "number_of_maps", "success_rate",
                "mean_expanded_nodes", "mean_generated_nodes",
                "mean_search_time_seconds", "mean_ground_truth_path_cost", "mean_path_cost_ratio",
            )
        ) + " |")
    report.extend([
        "", "## Artifacts", "",
        "- Raw results: final_evaluation/raw/search_results.csv",
        "- Per-map results: final_evaluation/per_map/per_map_comparison.csv",
        "- Aggregates and paired statistics: final_evaluation/aggregate/",
        "- Inference caches and manifest: final_evaluation/inference_caches/",
        "- Plots: final_evaluation/plots/",
        "- Complete-run ZIP: final_evaluation/phase_b_complete_run.zip",
        "",
        "Timeouts, exceptions, invalid paths, numerical failures, and memory failures remain in raw results.",
    ])
    (FINAL_ROOT / "final_evaluation_report.md").write_text(
        "\n".join(report) + "\n", encoding="utf-8"
    )

    zip_path = FINAL_ROOT / "phase_b_complete_run.zip"
    temp_zip = FINAL_ROOT / "phase_b_complete_run.zip.tmp"
    manifest_path = FINAL_ROOT / "bundle_manifest.json"
    manifest_files = []
    for path in sorted(RESULT_ROOT.rglob("*")):
        if not path.is_file() or path in {zip_path, temp_zip, manifest_path}:
            continue
        manifest_files.append({
            "path": path.relative_to(RESULT_ROOT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
    manifest_path.write_text(json.dumps({
        "run_id": RUN_ID, "run_variant": RUN_VARIANT,
        "includes_generated_datasets": True, "files": manifest_files,
    }, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    with zipfile.ZipFile(temp_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(RESULT_ROOT.rglob("*")):
            if not path.is_file() or path in {zip_path, temp_zip}:
                continue
            archive.write(path, path.relative_to(RESULT_ROOT).as_posix())
    temp_zip.replace(zip_path)
    zip_digest = sha256_file(zip_path)
    (FINAL_ROOT / "phase_b_complete_run.zip.sha256.json").write_text(
        json.dumps({
            "file": str(zip_path), "sha256": zip_digest,
            "manifest_file_count": len(manifest_files), "manifest": str(manifest_path),
        }, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    print(json.dumps(final_summary, indent=2))
    print(f"Complete-run ZIP: {zip_path}")
    print(f"ZIP SHA-256: {zip_digest}")

## End of notebook workflow

Sequential execution stops after ID-only integration by default because both approval flags are false. After separately approved pilot and final testing, the run contains generated data and manifests, checkpoints, histories, calibration, inference caches, raw/per-map/aggregate CSVs, plots, reports, and ZIP bundles. Use the run variant printed by the setup cell to identify this execution.